# K-Nearest Neighbors
### Definition
K-Nearest Neighbors is a supervised machine learning classification algorithm that looks at nearby training data to classify the next point.

This algorithm uses a provided distance function in order to classify the next point.

1. For the provided datapoint, calculate its distance between every training datapoint.
2. Then, sort the datapoints by distances, and store the $k$ closest datapoints to the provided point.
3. Classify the datapoint as the label that appears most among the $k$ closest datapoints.

### Advantages
- Works well for a variety of tasks, as the distance function can be applied to many datasets
- Works for nonlinear data
- Simple implementation
### Disadvantages
- For larger datasets, prediction can be slow (sorting by the training data each time can be costly)
- Doesn't work well for data with overlap

## Imports

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

import struct

sns.set_theme()

# Implementation

In [3]:
# some distance functions
def euclidean_distance(p, q):
    return np.sqrt((p - q) @ (p - q))

def manhattan_distance(p, q):
    return sum(p-q)

In [4]:
def get_k_nearest_neighbors(dist_func, train_X, train_y, point, k):
    # build distance dictionary
    distances = []
    for i in range(np.shape(train_X)[0]):
        dist = dist_func(train_X[i], point)
        distances.append([train_X[i], train_y[i], dist])

    # then sort by distance and return k closest points
    return sorted(distances, key=lambda x: x[-1])[:k]

def predict_k_nearest_neighbors(dist_func, train_x, train_y, point, k):
    nearest_neighbors = get_k_nearest_neighbors(dist_func, train_x, train_y, point, k)
    prediction = 0.0
    for i in range(k):
        neighbor = nearest_neighbors[i]
        neighbor_prediction = neighbor[1]
        prediction += neighbor_prediction / k

    # round prediction to closest value; 0 if < 0.5, 1 if >= 0.5
    return float(round(prediction))


In [5]:
def test_k_nearest_neighbors(dist_func, train_X, train_y, test_X, test_y, k):
    good_preds = 0
    for i in range(test_X.shape[0]):
        pred = predict_k_nearest_neighbors(dist_func, train_X, train_y, test_X[i, :], k)
        good_preds += 1 if pred == test_y[i] else 0
    good_preds /= test_X.shape[0]
    return good_preds

# Data
We will load in the Titanic data that we created in the preprocessing step:

In [6]:
data_dir = "/home/patrick/Projects/DataScienceAndMachineLearning/Data/Titanic-Dataset"
train_X = np.load(f"{data_dir}/train_X.npy")
train_y = np.load(f"{data_dir}/train_y.npy")
test_X = np.load(f"{data_dir}/test_X.npy")
test_y = np.load(f"{data_dir}/test_y.npy")
PCA_train_X = np.load(f"{data_dir}/PCA_train_X.npy")
PCA_train_y = np.load(f"{data_dir}/PCA_train_y.npy")
PCA_test_X = np.load(f"{data_dir}/PCA_test_X.npy")
PCA_test_y = np.load(f"{data_dir}/PCA_test_y.npy")
train_X

array([[ 3.,  0., 22., ...,  1.,  0.,  0.],
       [ 1.,  1., 38., ...,  0.,  1.,  0.],
       [ 3.,  1., 26., ...,  0.,  1.,  0.],
       ...,
       [ 3.,  0., 27., ...,  1.,  0.,  0.],
       [ 1.,  0., 42., ...,  1.,  0.,  0.],
       [ 3.,  0., 20., ...,  1.,  0.,  0.]], shape=(622, 14))

# Full-Dimension Data Analysis
Now, print table of different k values for the full-dimension test data with Euclidean distance:

In [7]:
print("  k | Successful prediction rate on test data with Euclidean distance")
print("----|----------------------------------------------------------------")
for k in range(1, 21):
    pred_rate = test_k_nearest_neighbors(euclidean_distance, train_X, train_y, test_X, test_y, k)
    print(f" {k:2d} | {pred_rate*100.0:3.2f}%")

  k | Successful prediction rate on test data with Euclidean distance
----|----------------------------------------------------------------
  1 | 67.42%
  2 | 73.78%
  3 | 74.16%
  4 | 74.16%
  5 | 73.78%
  6 | 73.41%
  7 | 75.28%
  8 | 74.53%
  9 | 75.28%
 10 | 73.41%
 11 | 75.28%
 12 | 73.78%
 13 | 76.40%
 14 | 74.16%
 15 | 74.91%
 16 | 73.78%
 17 | 75.28%
 18 | 76.78%
 19 | 77.90%
 20 | 75.28%


Now, print table of different k values for the full-dimension test data with Manhattan distance:

In [8]:
print("  k | Successful prediction rate on test data with Manhattan distance")
print("----|----------------------------------------------------------------")
for k in range(1, 21):
    pred_rate = test_k_nearest_neighbors(manhattan_distance, train_X, train_y, test_X, test_y, k)
    print(f" {k:2d} | {pred_rate*100.0:3.2f}%")

  k | Successful prediction rate on test data with Manhattan distance
----|----------------------------------------------------------------
  1 | 0.644
  2 | 0.644
  3 | 0.356
  4 | 0.356
  5 | 0.356
  6 | 0.356
  7 | 0.356
  8 | 0.356
  9 | 0.356
 10 | 0.356
 11 | 0.356
 12 | 0.356
 13 | 0.356
 14 | 0.356
 15 | 0.356
 16 | 0.356
 17 | 0.356
 18 | 0.356
 19 | 0.356
 20 | 0.356


We can see that Euclidean distance works much better for this data than manhattan distance, with a max successful prediction rate of 0.779, especially for high k values.

# PCA Data Analysis

Now, print table of different k values for the rank 2 PCA test data with Euclidean distance:

In [10]:
print("  k | Successful prediction rate on test data with Euclidean distance")
print("----|----------------------------------------------------------------")
for k in range(1, 21):
    pred_rate = test_k_nearest_neighbors(euclidean_distance, PCA_train_X, PCA_train_y, PCA_test_X, PCA_test_y, k)
    print(f" {k:2d} | {pred_rate*100.0:3.2f}%")

  k | Successful prediction rate on test data with Euclidean distance
----|----------------------------------------------------------------
  1 | 67.42%
  2 | 74.53%
  3 | 73.78%
  4 | 74.53%
  5 | 71.91%
  6 | 74.91%
  7 | 72.66%
  8 | 74.16%
  9 | 75.28%
 10 | 76.78%
 11 | 75.28%
 12 | 74.53%
 13 | 73.41%
 14 | 74.91%
 15 | 75.28%
 16 | 75.28%
 17 | 75.28%
 18 | 72.66%
 19 | 73.41%
 20 | 73.41%


We can see that, for Euclidean distance, the KNN algorithm works around as well when used with PCA data as with the full dimension dataset.